# Generate PDBs from pair_block_47.npy (77 proteins)

Generates PDB files from `{protein_id}_pair_block_47.npy` for all proteins under the base directory using the structure module.

**Layout:** `{base}/{protein_id}/{protein_id}_pair_block_47.npy`

**Optional inputs (same folder):** `single_block_47.npy`, `{id}.fasta` / `seq.fasta`, or `aatype_47.npy`

In [ ]:
# Imports and helpers
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from typing import List, Optional, Tuple

import numpy as np

LAYER = 47
PROJECT_ROOT = Path(".").resolve()


def discover_proteins(base: Path) -> List[Tuple[str, Path]]:
    """Return [(protein_id, pair_npy_path), ...] for all proteins with pair_block_47.npy."""
    pairs = []
    if not base.is_dir():
        return pairs
    for item in sorted(base.iterdir()):
        if not item.is_dir():
            continue
        protein_id = item.name
        pair_npy = item / f"{protein_id}_pair_block_{LAYER}.npy"
        if pair_npy.is_file():
            pairs.append((protein_id, pair_npy))
    return pairs


def find_fasta(subdir: Path, protein_id: str) -> Optional[Path]:
    for fn in [f"{protein_id}.fasta", "seq.fasta"]:
        p = subdir / fn
        if p.is_file():
            return p
    for f in subdir.iterdir():
        if f.suffix == ".fasta":
            return f
    return None

print("✅ Imports OK")

In [ ]:
# Configuration
BASE = os.environ.get("BASE", "/storage/scratch1/5/sshrestha304/Autoencoder/CompleteProteins")
OUTPUT_DIR = None  # None = {BASE}/pair_block_47_pdbs
MODEL_DEVICE = "cpu"  # or "cuda", "cuda:0"
LIMIT = None  # None = all proteins; set to e.g. 3 for testing
DRY_RUN = False  # True = print commands only, don't run

base = Path(BASE).resolve()
output_dir = Path(OUTPUT_DIR or str(base / "pair_block_47_pdbs")).resolve()
run_script = PROJECT_ROOT / "run_structure_module.py"

if not run_script.is_file():
    raise FileNotFoundError(f"run_structure_module.py not found at {run_script}")

print(f"Base: {base}")
print(f"Output: {output_dir}")
print(f"Device: {MODEL_DEVICE}")
print(f"Limit: {LIMIT or 'all'}")
print(f"Dry run: {DRY_RUN}")

In [ ]:
# Discover proteins
pairs = discover_proteins(base)
if not pairs:
    raise SystemExit(f"No pair_block_{LAYER}.npy found under {base}")

total = len(pairs)
if LIMIT:
    pairs = pairs[:LIMIT]
    print(f"Processing first {len(pairs)} of {total} proteins")
else:
    print(f"Found {total} proteins with pair_block_{LAYER}.npy")

output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Process each protein
ok = 0
fail = 0

for i, (protein_id, pair_npy) in enumerate(pairs, 1):
    subdir = pair_npy.parent
    single_npy = subdir / f"single_block_{LAYER}.npy"
    aatype_npy = subdir / f"aatype_{LAYER}.npy"
    fasta_path = find_fasta(subdir, protein_id)

    cmd = [
        sys.executable,
        str(run_script),
        str(pair_npy),
        str(output_dir),
        "--model-device",
        MODEL_DEVICE,
    ]
    if single_npy.is_file():
        cmd.extend(["--single-npy", str(single_npy)])
    else:
        cmd.append("--single-from-pair")

    temp_aatype = None
    if fasta_path:
        cmd.extend(["--fasta", str(fasta_path)])
    elif aatype_npy.is_file():
        cmd.extend(["--aatype-npy", str(aatype_npy)])
    else:
        pair_arr = np.load(pair_npy, allow_pickle=True)
        n_res = pair_arr.shape[0] if pair_arr.ndim >= 2 else pair_arr.shape[1]
        fd, temp_aatype = tempfile.mkstemp(suffix=".npy")
        os.close(fd)
        np.save(temp_aatype, np.zeros(n_res, dtype=np.int64))
        cmd.extend(["--aatype-npy", temp_aatype])

    try:
        if DRY_RUN:
            print(f"[{i}/{len(pairs)}] Would run: {' '.join(cmd)}")
            ok += 1
        else:
            print(f"[{i}/{len(pairs)}] {protein_id}...", end=" ", flush=True)
            result = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
            if result.returncode == 0:
                out_pdb = output_dir / f"{pair_npy.stem}_structure.pdb"
                if out_pdb.is_file():
                    print(f"✅ {out_pdb.name}")
                    ok += 1
                else:
                    print("⚠️ No PDB written")
                    fail += 1
            else:
                err = result.stderr[:200] if result.stderr else result.stdout[:200]
                print(f"❌ {err}")
                fail += 1
    except subprocess.TimeoutExpired:
        print("❌ Timeout")
        fail += 1
    except Exception as e:
        print(f"❌ {e}")
        fail += 1
    finally:
        if temp_aatype and os.path.exists(temp_aatype):
            try:
                os.unlink(temp_aatype)
            except OSError:
                pass

print(f"\nDone: {ok} OK, {fail} failed")